# La tercera vía en el sistema bipartidista norteamericano

### Proceso de extracción y análisis de las elecciones en EUA de 1992 a 2020

*El siguiente proyecto tiene como objetivo crear una base de datos con la información de las elecciones presidenciales de EUA, con la finalidad de detectar los patrones de descontento con el sistema bipartidista mediante una tercera vía*

*PENDIENTE VER CÓMO HACER MAS COLORIDO LOS MARKDOWNS PONIENDO IMAGENE


## Marco teórico del análisis

* #### Ley de duverger en sistema de partidos y la teoría del voto estratégico:

Maurice Duverger planteó que las reglas del juego electoral determinan qué partidos sobreviven y configuran el tipo de sistema partidista.

Particularmente en EUA, predomina el sistema bipartidista. En este país, la regla de elección es de la mayoría relativa con distritos unininominales, es decir: el ganador se lleva todo con obtener la mayoría de votos, sin necesidad de alcanzar un número específico de votos y dejando a la competencia sin ningún tipo de recompensa por su desempeño, o sea, sin representación. 

* #### Comportamiento electoral (voto útil vs voto protesta:

En la teoría de la elección pública señala que el votante promedio calcula su decisión con lógica estratégica: 

1. Vale más malo por conocido que bueno por conocer.
2. Voto útil por pocas posibilidades de victoria de la tercera vía

* #### Comportamiento electoral (voto útil vs voto protesta:

En la teoría de la elección pública señala que el votante promedio calcula su decisión con lógica estratégica: 

1. Vale más malo por conocido que bueno por conocer.
2. Voto útil por pocas posibilidades de victoria de la tercera vía

Bajo este esquema de comportamiento, se puede esperar que si hay un porcentaje considerable de votación por una tercera vía, haya una manifestación explícita de inconformidad, partiendo de la hipótesis que el sistema electoral y bipartidista de EUA premia sólo a los ganadores y racionaliza el voto en ese sentido.

#### Coyuntura vs estructura: sobre el tipo de inconformidad

* Coyuntura: fenómeno de corto plazo y de corta duración, puede capitalizar una circunstancia específica y capitalizar votos en determinada situación, pero no afecta el sistema bipartidista en larga escala.

* Estructura: fenómeno de largo plazo, arraigo y que pone de manifiesto una inconformidad con el sistema que se traduce en un comportamiento electoral divergente de la conformación bipartidaria. Se traduce en hábito político



## Metodología

**1. DIMENSIÓN DE FRECUENCIA (Mide la Inconformidad Estructural)**

*Pregunta:* ¿En cuántas elecciones (de 8) el estado superó el promedio es el porcentaje de voto alternativo?

*Propósito:* Detectar la constancia o el hábito de protesta.  

**2. DIMENSIÓN DE MAGNITUD (Mide la Intensidad del Voto)**

*Pregunta:* En promedio, ¿qué tan alto nacional del año?                                        

*Problema:* Un pico como 1992 infla el promedio (*mean*)

*Solución:* Usar la MEDIANA (*median*) como umbral base para borrar el ruido de los picos coyunturales.

### Para materializar este trabajo, el proyecto se divide en las siguientes fases:

**1. Primer bloque:** flujo de extracción, transformación y carga

* En esta etapa, se cargará el CSV que contiene los datos de las elecciones de EUA desde 1976 hasta 2020.
* Se filtrará el periodo 1990/2020, depurando columnas y preparando los agredados por estado/año

**2. Segundo bloque:** Carga en base de datos relacional SQL

* Crearemos la conexión sqlite3.connect ('elecciones_usa.db')
* Cargaremos la tabla limpia usando df.to_sql('resultados_electorales', ...).

**3. Tercer bloque:** Consultas analíticas en SQL

* Ejecutaremos consultas con SELECT, GROUP BY y agregaciones para extraer el Referente Nacional (incluyendo la mediana), así como la Matriz de Frecuencia y la Magnitud.

## 1. Etapa ETL: Procesamiento e inserción en SQLite

In [1]:
import sqlite3
import pandas as pd

In [2]:
#Primer paso. Vamos a cargar el dataset desde la carpeta de entrada de Kaggle
df_raw = pd.read_csv("/kaggle/input/datasets/tunguz/us-elections-dataset/1976-2020-president.csv")

#Segundo paso. Vamos a ver las primeras 5 filas para inspección rutinaria
df_raw.head (5)

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
0,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"CARTER, JIMMY",DEMOCRAT,False,659170,1182850,20210113,NaN,DEMOCRAT
1,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,504070,1182850,20210113,NaN,REPUBLICAN
2,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"MADDOX, LESTER",AMERICAN INDEPENDENT PARTY,False,9198,1182850,20210113,NaN,OTHER
3,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"BUBAR, BENJAMIN """"BEN""""",PROHIBITION,False,6669,1182850,20210113,NaN,OTHER
4,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"HALL, GUS",COMMUNIST PARTY USE,False,1954,1182850,20210113,NaN,OTHER


In [3]:
#Tercer paso. Filtramos información, ya que el dataset original tiene datos desde 1976 y nuestro estudio empieza desde 1990

df_filtered = df_raw[df_raw["year"] >= 1990].copy()

#Cuarto paso. Verificamos cuáles años quedaron en nuestro dataset
print("Años incluidos en la muestra:")
print(sorted(df_filtered["year"].unique()))

Años incluidos en la muestra:
[np.int64(1992), np.int64(1996), np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020)]


In [4]:
#Cuarto paso. Crear una nueva columna homologando los partidos
df_filtered["party_simplified"] = df_filtered["party_detailed"].apply(
    lambda x: (
        x if x in ["DEMOCRAT", "REPUBLICAN"] else "THIRD_PARTY_OTHER"
    )
)

#Quinto paso. Ver cuántos registros quedaron por cada categoría
print(df_filtered["party_simplified"].value_counts())

party_simplified
THIRD_PARTY_OTHER    2166
REPUBLICAN            409
DEMOCRAT              408
Name: count, dtype: int64


In [5]:
#Sexto paso. Agrupar y sumar los votos por Año, Estado y Categoria de Partido
df_grouped = (
    df_filtered.groupby(
        [
            "year",
            "state",
            "state_po",
            "state_fips",
            "party_simplified",
            "totalvotes",
        ],
        as_index=False,
    )["candidatevotes"]
    .sum()
)

#Séptimo paso. Calcular el porcentaje que representa cada categoría sobre el total del estado
df_grouped["vote_pct"] = (
    df_grouped["candidatevotes"] / df_grouped["totalvotes"]
) * 100

#Octavo paso. Mostrar las primeras 10 filas de la tabla procesada
df_grouped.head(10)

,year,state,state_po,state_fips,party_simplified,totalvotes,candidatevotes,vote_pct
0,1992,ALABAMA,AL,1,DEMOCRAT,1688060,690080,40.880064
1,1992,ALABAMA,AL,1,REPUBLICAN,1688060,804283,47.645404
2,1992,ALABAMA,AL,1,THIRD_PARTY_OTHER,1688060,193697,11.474533
3,1992,ALASKA,AK,2,DEMOCRAT,258506,78294,30.287111
4,1992,ALASKA,AK,2,REPUBLICAN,258506,102000,39.457498
5,1992,ALASKA,AK,2,THIRD_PARTY_OTHER,258506,78212,30.255391
6,1992,ARIZONA,AZ,4,DEMOCRAT,1486975,543050,36.520453
7,1992,ARIZONA,AZ,4,REPUBLICAN,1486975,572086,38.473142
8,1992,ARIZONA,AZ,4,THIRD_PARTY_OTHER,1486975,371839,25.006406
9,1992,ARKANSAS,AR,5,DEMOCRAT,950653,505823,53.207953
